# 🔍 Random Forest Hyperparameter Search
### Grid Search vs Random Search — Credit Card Fraud Detection
Uses your existing SMOTE + RF pipeline, evaluating on **PR-AUC** (best metric for imbalanced fraud data).

## 1. Load & Preprocess Data

In [1]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.model_selection import train_test_split

# Load dataset
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    'mlg-ulb/creditcardfraud',
    'creditcard.csv'
)

# Preprocessing (matching your notebook exactly)
new_df = df.copy()
new_df['Amount'] = RobustScaler().fit_transform(new_df['Amount'].to_numpy().reshape(-1, 1))
new_df['Time']   = StandardScaler().fit_transform(new_df[['Time']])
new_df = new_df.sample(frac=1, random_state=42)

# Train / Test / Val split (80 / 10 / 10 — stratified)
train, temp = train_test_split(new_df, test_size=0.2, stratify=new_df['Class'], random_state=42)
test,  val  = train_test_split(temp,   test_size=0.5, stratify=temp['Class'],   random_state=42)

x_train = train.drop(columns=['Class']); y_train = train['Class']
x_test  = test.drop(columns=['Class']);  y_test  = test['Class']
x_val   = val.drop(columns=['Class']);   y_val   = val['Class']

# Use top-20 important features (from your notebook)
from sklearn.ensemble import RandomForestClassifier
rf_tmp = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_tmp.fit(x_train, y_train)
sorted_idx       = np.argsort(rf_tmp.feature_importances_)[::-1]
important_features = x_train.columns[sorted_idx[:20]]

x_train_imp = x_train[important_features]
x_val_imp   = x_val[important_features]
x_test_imp  = x_test[important_features]

print(f'Train: {x_train_imp.shape}  |  Val: {x_val_imp.shape}  |  Test: {x_test_imp.shape}')
print(f'Fraud rate — train: {y_train.mean():.4%}  |  val: {y_val.mean():.4%}')
print(f'Top features: {list(important_features[:10])}')

Train: (227845, 20)  |  Val: (28481, 20)  |  Test: (28481, 20)
Fraud rate — train: 0.1729%  |  val: 0.1720%
Top features: ['V17', 'V14', 'V12', 'V16', 'V10', 'V11', 'V9', 'V4', 'V18', 'V7']


## 2. Define the SMOTE + RF Pipeline & Parameter Grids

In [2]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

# Base pipeline (matches your best experiment)
base_pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('rf',    RandomForestClassifier(random_state=42, n_jobs=-1))
])

# ── Grid Search param grid (exhaustive) ──────────────────────────────────────
# Focused around your known best params to keep it tractable
grid_params = {
    'smote__sampling_strategy': [0.2, 0.3, 0.5],
    'smote__k_neighbors':       [3, 5, 7],
    'rf__n_estimators':         [100, 200, 300],
    'rf__max_depth':            [8, 10, None],
    'rf__min_samples_leaf':     [1, 2, 4],
}
# 3 × 3 × 3 × 3 × 3 = 243 combinations × 3 folds = 729 fits
# We'll use a focused subset for Grid Search to keep runtime reasonable:
grid_params_focused = {
    'smote__sampling_strategy': [0.2, 0.3, 0.5],
    'smote__k_neighbors':       [3, 5],
    'rf__n_estimators':         [200, 300],
    'rf__max_depth':            [None, 10],
    'rf__min_samples_leaf':     [1, 2, 4],
}
# 3 × 2 × 2 × 2 × 3 = 72 combos × 3 folds = 216 fits

# ── Random Search param distribution (wider exploration) ─────────────────────
from scipy.stats import randint, uniform
random_params = {
    'smote__sampling_strategy': [0.1, 0.2, 0.3, 0.4, 0.5],
    'smote__k_neighbors':       [3, 5, 7, 9],
    'rf__n_estimators':         randint(100, 500),
    'rf__max_depth':            [6, 8, 10, 12, 15, None],
    'rf__min_samples_leaf':     [1, 2, 4, 8],
    'rf__max_features':         ['sqrt', 'log2', 0.5, 0.7],
    'rf__min_samples_split':    randint(2, 20),
}

print('Grid Search  — focused grid defined ✓')
print('Random Search — wider distribution defined ✓')

Grid Search  — focused grid defined ✓
Random Search — wider distribution defined ✓


## 3. Run Random Search (broad exploration first)

In [16]:
from sklearn.model_selection import RandomizedSearchCV

print('Running RandomizedSearchCV (n_iter=30, cv=3)...')
t0 = time.time()

random_search = RandomizedSearchCV(
    base_pipeline,
    param_distributions=random_params,
    n_iter=20,                      # 30 random combos
    scoring='recall',    # PR-AUC — best for imbalanced data
    cv=2,
    n_jobs=-1,
    random_state=42,
    verbose=1,
    return_train_score=True
)

random_search.fit(x_train_imp, y_train)

random_time = time.time() - t0
print(f'\n✅ Random Search done in {random_time:.1f}s')
print(f'   Best CV PR-AUC : {random_search.best_score_:.4f}')
print(f'   Best params    : {random_search.best_params_}')

Running RandomizedSearchCV (n_iter=30, cv=3)...
Fitting 2 folds for each of 20 candidates, totalling 40 fits

✅ Random Search done in 810.7s
   Best CV PR-AUC : 0.8553
   Best params    : {'rf__max_depth': 6, 'rf__max_features': 0.5, 'rf__min_samples_leaf': 2, 'rf__min_samples_split': 5, 'rf__n_estimators': 188, 'smote__k_neighbors': 9, 'smote__sampling_strategy': 0.2}


In [6]:
from sklearn.model_selection import RandomizedSearchCV

print('Running RandomizedSearchCV (n_iter=30, cv=3)...')
t0 = time.time()

random_search = RandomizedSearchCV(
    base_pipeline,
    param_distributions=random_params,
    n_iter=10,                      # 30 random combos
    scoring='recall',    # PR-AUC — best for imbalanced data
    cv=2,
    n_jobs=-1,
    random_state=42,
    verbose=1,
    return_train_score=True
)

random_search.fit(x_train_imp, y_train)

random_time = time.time() - t0
print(f'\n✅ Random Search done for recall {random_time:.1f}s')
print(f'   Best CV PR-AUC : {random_search.best_score_:.4f}')
print(f'   Best params    : {random_search.best_params_}')

Running RandomizedSearchCV (n_iter=30, cv=3)...
Fitting 2 folds for each of 10 candidates, totalling 20 fits

✅ Random Search done for recall 430.4s
   Best CV PR-AUC : 0.8553
   Best params    : {'rf__max_depth': 6, 'rf__max_features': 0.5, 'rf__min_samples_leaf': 2, 'rf__min_samples_split': 5, 'rf__n_estimators': 188, 'smote__k_neighbors': 9, 'smote__sampling_strategy': 0.2}


In [20]:
# Use the GRID SEARCH best params (not the random search ones from before)
randomforest_tuned_for_recall = Pipeline([
    ('smote', SMOTE(
        sampling_strategy=0.2,
        k_neighbors=3,
        random_state=42
    )),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

randomforest_tuned_for_recall.fit(x_train_imp, y_train)

,steps,"[('smote', ...), ('rf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,sampling_strategy,0.2
,random_state,42
,k_neighbors,3
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2


In [21]:
from sklearn.metrics import classification_report, average_precision_score, precision_recall_curve

y_pred = randomforest_tuned_for_recall.predict(x_val_imp)
print(classification_report(y_val, y_pred))
print("PR-AUC:", average_precision_score(y_val, y_pred))

# Threshold tuning by code: sweep all thresholds, pick the one that maximizes chosen metric
y_probs = randomforest_tuned_for_recall.predict_proba(x_val_imp)[:, 1]
precision, recall_curve, thresholds = precision_recall_curve(y_val, y_probs)
# For each threshold we get one (P,R) pair: thresholds[j] -> (precision[j+1], recall[j+1])
n = len(thresholds)
eps = 1e-9

def f_beta(p, r, beta=1):
    return (1 + beta**2) * p * r / (beta**2 * p + r + eps)

# Compute F1 and F2 (recall-weighted) for every threshold
f1_scores = np.array([f_beta(precision[j+1], recall_curve[j+1], 1) for j in range(n)])
f2_scores = np.array([f_beta(precision[j+1], recall_curve[j+1], 2) for j in range(n)])

# Choose threshold that maximizes F1 (balanced)
best_j = np.argmax(f1_scores)
best_thr = thresholds[best_j]
y_pred_tuned = (y_probs >= best_thr).astype(int)
print("\n--- Threshold tuning (maximize F1 on validation) ---")
print(f"Best threshold = {best_thr:.4f}  →  P = {precision[best_j+1]:.4f}, R = {recall_curve[best_j+1]:.4f}, F1 = {f1_scores[best_j]:.4f}")
print(classification_report(y_val, y_pred_tuned))
print("PR-AUC (tuned):", average_precision_score(y_val, y_pred_tuned))

# Optional: threshold that maximizes F2 (more recall-oriented, still data-driven)
best_j_f2 = np.argmax(f2_scores)
best_thr_f2 = thresholds[best_j_f2]
y_pred_f2 = (y_probs >= best_thr_f2).astype(int)
print("\n--- Alternative: threshold that maximizes F2 (favors recall) ---")
print(f"Best threshold = {best_thr_f2:.4f}  →  P = {precision[best_j_f2+1]:.4f}, R = {recall_curve[best_j_f2+1]:.4f}, F2 = {f2_scores[best_j_f2]:.4f}")
print(classification_report(y_val, y_pred_f2))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28432
           1       0.89      0.84      0.86        49

    accuracy                           1.00     28481
   macro avg       0.95      0.92      0.93     28481
weighted avg       1.00      1.00      1.00     28481

PR-AUC: 0.7460661596437196

--- Threshold tuning (maximize F1 on validation) ---
Best threshold = 0.5485  →  P = 0.9111, R = 0.8367, F1 = 0.8723
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28432
           1       0.89      0.84      0.86        49

    accuracy                           1.00     28481
   macro avg       0.95      0.92      0.93     28481
weighted avg       1.00      1.00      1.00     28481

PR-AUC (tuned): 0.7460661596437196

--- Alternative: threshold that maximizes F2 (favors recall) ---
Best threshold = 0.5485  →  P = 0.9111, R = 0.8367, F2 = 0.8506
              precision    recall  f

## 4. Why recall was stuck & how to get ~0.87

- **F1-optimal threshold** favors balance, not high recall, so you were getting ~0.82 recall.
- **Target-recall threshold**: pick the threshold that gives **recall ≥ 0.87** (above). That will lower the bar for “fraud” and usually trade some precision for recall.
- If the current model **can’t** reach 0.87 at any threshold, try a more recall-oriented model below (`class_weight='balanced'` and/or higher SMOTE).

In [13]:
# Optional pipeline: threshold tuned by code (maximize F1 on validation)
from sklearn.metrics import precision_recall_curve

pipe_balanced = Pipeline([
    ('smote', SMOTE(sampling_strategy=0.28, k_neighbors=5, random_state=42)),
    ('rf', RandomForestClassifier(
        n_estimators=200, max_depth=8, max_features=0.5,
        min_samples_leaf=2, min_samples_split=5,
        class_weight='balanced',
        random_state=42, n_jobs=-1
    ))
])
pipe_balanced.fit(x_train_imp, y_train)

y_probs_hr = pipe_balanced.predict_proba(x_val_imp)[:, 1]
precision_hr, recall_hr, thresholds_hr = precision_recall_curve(y_val, y_probs_hr)
n_hr = len(thresholds_hr)
eps = 1e-9
f1_hr = np.array([2 * precision_hr[j+1] * recall_hr[j+1] / (precision_hr[j+1] + recall_hr[j+1] + eps) for j in range(n_hr)])
best_j = np.argmax(f1_hr)
thr = thresholds_hr[best_j]
y_pred_tuned = (y_probs_hr >= thr).astype(int)
print("Pipeline (SMOTE 0.28 + class_weight) — threshold = argmax F1 on validation:")
print(f"Best threshold = {thr:.4f}  →  P = {precision_hr[best_j+1]:.4f}, R = {recall_hr[best_j+1]:.4f}, F1 = {f1_hr[best_j]:.4f}")
print(classification_report(y_val, y_pred_tuned))

Pipeline (SMOTE 0.28 + class_weight) — threshold = argmax F1 on validation:
Best threshold = 0.8514  →  P = 0.8636, R = 0.7755, F1 = 0.8172
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28432
           1       0.84      0.78      0.81        49

    accuracy                           1.00     28481
   macro avg       0.92      0.89      0.90     28481
weighted avg       1.00      1.00      1.00     28481



## 5. More experiments for better recall

We run several pipeline variants (SMOTE ratio, class_weight, tree depth), tune the decision threshold by maximizing F1 or F2 on validation, and compare validation metrics. Then we try all 30 features and optional random search with F1 scoring.

In [14]:
# Experiment 1: Multiple pipeline configs — same threshold tuning (F1 and F2)
from sklearn.metrics import precision_recall_curve, classification_report, f1_score, recall_score, precision_score

def eval_pipeline(pipe, x_tr, y_tr, x_val, y_val, name):
    pipe.fit(x_tr, y_tr)
    probs = pipe.predict_proba(x_val)[:, 1]
    prec, rec, thrs = precision_recall_curve(y_val, probs)
    n = len(thrs)
    eps = 1e-9
    f1 = np.array([2 * prec[j+1] * rec[j+1] / (prec[j+1] + rec[j+1] + eps) for j in range(n)])
    f2 = np.array([5 * prec[j+1] * rec[j+1] / (4 * prec[j+1] + rec[j+1] + eps) for j in range(n)])
    # F1-optimal
    j1 = np.argmax(f1)
    t1 = thrs[j1]
    p1 = (probs >= t1).astype(int)
    # F2-optimal (favors recall)
    j2 = np.argmax(f2)
    t2 = thrs[j2]
    p2 = (probs >= t2).astype(int)
    return {
        'name': name,
        'thr_f1': t1, 'thr_f2': t2,
        'prec_f1': precision_score(y_val, p1, zero_division=0), 'rec_f1': recall_score(y_val, p1, zero_division=0), 'f1_f1': f1_score(y_val, p1, zero_division=0),
        'prec_f2': precision_score(y_val, p2, zero_division=0), 'rec_f2': recall_score(y_val, p2, zero_division=0), 'f1_f2': f1_score(y_val, p2, zero_division=0),
        'pipe': pipe,
        'probs': probs, 'thrs': thrs, 'prec': prec, 'rec': rec,
    }

configs = [
    ('SMOTE 0.2, RF default', Pipeline([
        ('smote', SMOTE(sampling_strategy=0.2, k_neighbors=5, random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=200, max_depth=10, max_features='sqrt', random_state=42, n_jobs=-1))
    ])),
    ('SMOTE 0.35, RF default', Pipeline([
        ('smote', SMOTE(sampling_strategy=0.35, k_neighbors=5, random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=200, max_depth=10, max_features='sqrt', random_state=42, n_jobs=-1))
    ])),
    ('SMOTE 0.2, class_weight=balanced', Pipeline([
        ('smote', SMOTE(sampling_strategy=0.2, k_neighbors=5, random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=200, max_depth=10, max_features='sqrt', class_weight='balanced', random_state=42, n_jobs=-1))
    ])),
    ('SMOTE 0.35, class_weight=balanced', Pipeline([
        ('smote', SMOTE(sampling_strategy=0.35, k_neighbors=5, random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=200, max_depth=10, max_features='sqrt', class_weight='balanced', random_state=42, n_jobs=-1))
    ])),
    ('SMOTE 0.5, class_weight=balanced, depth=12', Pipeline([
        ('smote', SMOTE(sampling_strategy=0.5, k_neighbors=5, random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=250, max_depth=12, max_features='sqrt', class_weight='balanced', random_state=42, n_jobs=-1))
    ])),
]

results = []
for name, pipe in configs:
    r = eval_pipeline(pipe, x_train_imp, y_train, x_val_imp, y_val, name)
    results.append(r)

# Summary table (fraud class: precision, recall, F1) for F1-optimal and F2-optimal threshold
print('Validation metrics (fraud class) — F1-optimal threshold:')
print(pd.DataFrame([{'config': r['name'], 'P': r['prec_f1'], 'R': r['rec_f1'], 'F1': r['f1_f1'], 'thr': r['thr_f1']} for r in results]).to_string(index=False))
print()
print('Validation metrics (fraud class) — F2-optimal threshold (favors recall):')
print(pd.DataFrame([{'config': r['name'], 'P': r['prec_f2'], 'R': r['rec_f2'], 'F1': r['f1_f2'], 'thr': r['thr_f2']} for r in results]).to_string(index=False))

# Best by recall (F2 run) and by F1 (F1 run)
best_recall_idx = np.argmax([r['rec_f2'] for r in results])
best_f1_idx = np.argmax([r['f1_f1'] for r in results])
print()
print(f"Best recall (F2 threshold): {results[best_recall_idx]['name']} — R={results[best_recall_idx]['rec_f2']:.4f}, P={results[best_recall_idx]['prec_f2']:.4f}, F1={results[best_recall_idx]['f1_f2']:.4f}")
print(f"Best F1 (F1 threshold):     {results[best_f1_idx]['name']} — R={results[best_f1_idx]['rec_f1']:.4f}, P={results[best_f1_idx]['prec_f1']:.4f}, F1={results[best_f1_idx]['f1_f1']:.4f}")
best_pipeline = results[best_recall_idx]['pipe']  # use best-recall model for later
best_threshold_f2 = results[best_recall_idx]['thr_f2']

Validation metrics (fraud class) — F1-optimal threshold:
                                    config        P        R       F1      thr
                     SMOTE 0.2, RF default 0.891304 0.836735 0.863158 0.675227
                    SMOTE 0.35, RF default 0.888889 0.816327 0.851064 0.666548
          SMOTE 0.2, class_weight=balanced 0.883721 0.775510 0.826087 0.758863
         SMOTE 0.35, class_weight=balanced 0.880952 0.755102 0.813187 0.790760
SMOTE 0.5, class_weight=balanced, depth=12 0.880952 0.755102 0.813187 0.747041

Validation metrics (fraud class) — F2-optimal threshold (favors recall):
                                    config        P        R       F1      thr
                     SMOTE 0.2, RF default 0.891304 0.836735 0.863158 0.675227
                    SMOTE 0.35, RF default 0.854167 0.836735 0.845361 0.561247
          SMOTE 0.2, class_weight=balanced 0.883721 0.775510 0.826087 0.758863
         SMOTE 0.35, class_weight=balanced 0.829787 0.795918 0.812500 0.722265


In [15]:
# Experiment 2: Random search with class_weight in grid and scoring='f1'
from sklearn.model_selection import RandomizedSearchCV

params_with_weight = {**random_params, 'rf__class_weight': [None, 'balanced']}
pipe_f1 = RandomizedSearchCV(
    base_pipeline,
    param_distributions=params_with_weight,
    n_iter=15,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)
pipe_f1.fit(x_train_imp, y_train)
print('Best F1 CV score:', pipe_f1.best_score_)
print('Best params:', pipe_f1.best_params_)

# Evaluate best estimator with threshold tuning on val
best_est = pipe_f1.best_estimator_
probs_f1 = best_est.predict_proba(x_val_imp)[:, 1]
prec, rec, thrs = precision_recall_curve(y_val, probs_f1)
n = len(thrs)
f1_scores = np.array([2 * prec[j+1] * rec[j+1] / (prec[j+1] + rec[j+1] + 1e-9) for j in range(n)])
j_best = np.argmax(f1_scores)
thr_best = thrs[j_best]
pred_f1 = (probs_f1 >= thr_best).astype(int)
print('\nBest Random Search model (F1-optimal threshold on val):')
print(classification_report(y_val, pred_f1))
print(f'Threshold = {thr_best:.4f}')

Fitting 3 folds for each of 15 candidates, totalling 45 fits
Best F1 CV score: 0.8173008727996036
Best params: {'rf__class_weight': None, 'rf__max_depth': None, 'rf__max_features': 0.7, 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 17, 'rf__n_estimators': 336, 'smote__k_neighbors': 5, 'smote__sampling_strategy': 0.5}

Best Random Search model (F1-optimal threshold on val):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28432
           1       0.88      0.78      0.83        49

    accuracy                           1.00     28481
   macro avg       0.94      0.89      0.91     28481
weighted avg       1.00      1.00      1.00     28481

Threshold = 0.7061


In [ ]:
# Experiment 3: Use all 30 features with SMOTE 0.35 + class_weight=balanced
pipe_allfeat = Pipeline([
    ('smote', SMOTE(sampling_strategy=0.35, k_neighbors=5, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=200, max_depth=10, max_features='sqrt', class_weight='balanced', random_state=42, n_jobs=-1))
])
pipe_allfeat.fit(x_train, y_train)

probs_af = pipe_allfeat.predict_proba(x_val)[:, 1]
prec_af, rec_af, thrs_af = precision_recall_curve(y_val, probs_af)
n_af = len(thrs_af)
f1_af = np.array([2 * prec_af[j+1] * rec_af[j+1] / (prec_af[j+1] + rec_af[j+1] + 1e-9) for j in range(n_af)])
f2_af = np.array([5 * prec_af[j+1] * rec_af[j+1] / (4 * prec_af[j+1] + rec_af[j+1] + 1e-9) for j in range(n_af)])
j_f1 = np.argmax(f1_af)
j_f2 = np.argmax(f2_af)
pred_af_f1 = (probs_af >= thrs_af[j_f1]).astype(int)
pred_af_f2 = (probs_af >= thrs_af[j_f2]).astype(int)

print('All 30 features — F1-optimal threshold:')
print(classification_report(y_val, pred_af_f1))
print('All 30 features — F2-optimal threshold (favors recall):')
print(classification_report(y_val, pred_af_f2))

In [ ]:
# Final: refit best model from Experiment 1 and show full report (F2 threshold for better recall)
best_pipeline.fit(x_train_imp, y_train)
y_probs_final = best_pipeline.predict_proba(x_val_imp)[:, 1]
y_pred_final = (y_probs_final >= best_threshold_f2).astype(int)
print('Final recommended model (best recall config + F2-optimal threshold):')
print(classification_report(y_val, y_pred_final))
print('Threshold =', best_threshold_f2)

In [17]:
from sklearn.model_selection import GridSearchCV

print('Running GridSearchCV (focused grid, cv=3)...')
t0 = time.time()

grid_search = GridSearchCV(
    base_pipeline,
    param_grid=grid_params_focused,
    scoring='average_precision',
    cv=2,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_search.fit(x_train_imp, y_train)

grid_time = time.time() - t0
print(f'\n✅ Grid Search done in {grid_time:.1f}s')
print(f'   Best CV PR-AUC : {grid_search.best_score_:.4f}')
print(f'   Best params    : {grid_search.best_params_}')

Running GridSearchCV (focused grid, cv=3)...
Fitting 2 folds for each of 72 candidates, totalling 144 fits

✅ Grid Search done in 2106.4s
   Best CV PR-AUC : 0.8533
   Best params    : {'rf__max_depth': None, 'rf__min_samples_leaf': 2, 'rf__n_estimators': 300, 'smote__k_neighbors': 3, 'smote__sampling_strategy': 0.2}


In [18]:
#train a new model with the best params
best_pipeline = Pipeline([
    ('smote', SMOTE(sampling_strategy=0.2, k_neighbors=3, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=300, max_depth=None, min_samples_leaf=2, random_state=42, n_jobs=-1))
])
best_pipeline.fit(x_train_imp, y_train)


,steps,"[('smote', ...), ('rf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,sampling_strategy,0.2
,random_state,42
,k_neighbors,3
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2


In [19]:
#predict on the validation  set
y_pred = best_pipeline.predict(x_val_imp)

#print the classification report
print(classification_report(y_val, y_pred))

#threshold tuning
from sklearn.metrics import precision_recall_curve
y_probs = best_pipeline.predict_proba(x_val_imp)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_val, y_probs)
f1_scores = 2 * precision * recall / (precision + recall + 1e-9)
best_idx  = np.argmax(f1_scores)
best_thr  = thresholds[best_idx]
y_pred_tuned    = (y_probs >= best_thr).astype(int)

print("Classification report -- threshold tuned!")
print(classification_report(y_val, y_pred_tuned))
pr_auc_tuned = average_precision_score(y_val, y_pred_tuned)
print("PR-AUC tuned:", pr_auc_tuned)





              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28432
           1       0.89      0.84      0.86        49

    accuracy                           1.00     28481
   macro avg       0.95      0.92      0.93     28481
weighted avg       1.00      1.00      1.00     28481

Classification report -- threshold tuned!
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28432
           1       0.91      0.84      0.87        49

    accuracy                           1.00     28481
   macro avg       0.96      0.92      0.94     28481
weighted avg       1.00      1.00      1.00     28481

PR-AUC tuned: 0.7626391656577194
